# LoopMoE 1B (Test Branch) — Colab TPU v6e-1 MFU & HBM Benchmark

Measures steady-state **MFU%**, **tokens/sec**, **XLA compile time**, and **peak HBM** at uniform **4K (`R=1`)** (`V=32,768`, 2-Stream Closed-Form mHC, `128×1024 top-5 + 1 shared×1024`, Per-Loop IterAdaLN, Pre-Loop `4×256` Engram, Cubic5 Muon, single-pass fused 16K CE) with **Tokamax Megablox `ragged_dot`** + **Splash Attention** in native **BF16**.

In [ ]:
# 1. Clone/update Test branch and install latest pinned stack from requirements.txt
import os
REPO_DIR = '/content/M3_Test'
if not os.path.exists(REPO_DIR):
    !git clone -b Test https://github.com/GarD0m/M3.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin Test

!pip install -q -U -r {REPO_DIR}/requirements.txt

In [ ]:
# 2. Probe TPU in a subprocess so the Jupyter kernel does NOT lock libtpu.so
!python -c "import jax; print('Backend:', jax.default_backend(), '| Devices:', jax.devices()); [print('Memory:', d.memory_stats()) for d in jax.local_devices()]"

In [ ]:
# 3. Run full 1B MFU & HBM benchmark (4K R=1)
%cd {REPO_DIR}
!python scripts/bench_tpu_mfu.py \
    --tpu-preset v6e-1 \
    --seq-lens 4096 \
    --loops 1 \
    --microbatch 16 \
    --grad-accum 4 \
    --chunk-ce-size 16384 \
    --warmup-steps 2 \
    --bench-steps 5

In [ ]:
# 4. Optional: test grad_accum=8 with microbatch=16 at 4K R=1
%cd {REPO_DIR}
!python scripts/bench_tpu_mfu.py \
    --tpu-preset v6e-1 \
    --seq-lens 4096 \
    --loops 1 \
    --microbatch 16 \
    --grad-accum 8 \
    --chunk-ce-size 16384 \
    --warmup-steps 1 \
    --bench-steps 3